In [1]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Recreate the same architecture
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

# 2. Load saved weights
checkpoint = torch.load(
    "models/resnet18_head_only_best.pt",
    map_location=device,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Classes:", checkpoint["class_names"])
print("Best validation AUC:", checkpoint["best_val_auc"])

Classes: ['NORMAL', 'Hemorrhagic']
Best validation AUC: 0.7518512189564821


In [2]:
from pathlib import Path

IMAGE_PATH = Path(
    r"E:\vscode\Vision model practice\data\Data\NORMAL\N6[N6]\N6_0_100.jpg"
)

print("Image exists:", IMAGE_PATH.exists())

Image exists: True


In [3]:
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

image = Image.open(IMAGE_PATH).convert("RGB")
image_tensor = resnet_transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(image_tensor)
    probabilities = torch.softmax(outputs, dim=1)[0]

prediction_index = probabilities.argmax().item()
prediction = checkpoint["class_names"][prediction_index]

print("Prediction:", prediction)
print("NORMAL probability:", probabilities[0].item())
print("Hemorrhagic probability:", probabilities[1].item())

Prediction: Hemorrhagic
NORMAL probability: 0.4829719364643097
Hemorrhagic probability: 0.5170280337333679
